In [1]:
import gurobipy as grb
import numpy as np
import timeit
import scipy
from sklearn.datasets import make_classification
import sklearn
from ucimlrepo import fetch_ucirepo 
import math
from sklearn import metrics
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
import copy
from sklearn.utils import shuffle
from sklearn.preprocessing import OneHotEncoder
from collections import Counter
from sklearn.datasets import fetch_openml
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import pandas as pd
from sklearn.utils import shuffle
from sklearn.preprocessing import MinMaxScaler

In [2]:
# Setting Up Data Generation  

def encode_labels(y_all):
    # Automatically generate the label mapping from unique labels
    unique_labels = sorted(np.unique(y_all))  # Sort to ensure contiguous mapping
    label_mapping = {label: idx for idx, label in enumerate(unique_labels)}

    # Encode the labels using the generated mapping
    return np.array([label_mapping[label] for label in y_all]), label_mapping


def generate_n_sample_vec(y_all, perc_test):
    unique_classes, class_counts = np.unique(y_all, return_counts=True)

    # Compute number of test samples per class
    n_sample = np.floor(class_counts * perc_test).astype(int)

    # Ensure at least 1 test sample for each class 
    for i, count in enumerate(class_counts):
        if n_sample[i] == 0 and count > 0:
            n_sample[i] = 1

    # Compute training samples per class 
    n_train_rem = class_counts - n_sample

    # Create dictionaries for train and test samples per class
    n_train = {unique_classes[i]: n_train_rem[i] for i in range(len(unique_classes))}
    n_test = {unique_classes[i]: n_sample[i] for i in range(len(unique_classes))}

    print(f" n_train: {n_train}, n_test: {n_test}")
    return n_train, n_test


def l2_normalize_rows(X, eps=1e-12):
    
    X = np.asarray(X, dtype=float)
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    return X / np.maximum(norms, eps)


def drop_nonfinite_rows(X, y=None, name="data", verbose=True):

    X = np.asarray(X, dtype=float)
    mask = np.isfinite(X).all(axis=1)  # True if row has no NaN/Inf

    dropped = int((~mask).sum())
    if verbose and dropped > 0:
        print(f"[{name}] Dropping {dropped} / {len(X)} rows with NaN/Inf")

    X2 = X[mask]
    if y is None:
        return X2
    y2 = np.asarray(y)[mask]
    return X2, y2


def class_split_per_client(train_class_counts, perc_class_per_client, G, client_sample_props):
    
    #Get class counts
    n_classes = len(train_class_counts)
    
    #Case 1: User-defined table of percentages per client and per class
    if isinstance(perc_class_per_client, np.ndarray):
        print("Using user-defined percentages")
        if perc_class_per_client.shape != (G, n_classes):
            raise ValueError(f"The shape of `perc_class_per_client` should be ({G}, {n_classes})")

        # Normalize to ensure each row sums to 1
        row_sums = perc_class_per_client.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1  # Prevent division by zero
        perc_class_per_client = perc_class_per_client / row_sums
        
        use_client_sample_props = True
        if client_sample_props is None:
            client_sample_props = [1 / G] * G
        
    # Case 2: Randomly generated percentages for each client
    elif perc_class_per_client == -1:
        
        if G == 1:
            perc_class_per_client = {0: {class_label: 1.0 for class_label in train_class_counts}}  # Single client gets all
        
        else:
            if G == 2:
                perc_class_per_client = {}
                for class_label in train_class_counts:
                    r = np.random.uniform(low=0.1, high=0.9)
                    perc_class_per_client[0] = perc_class_per_client.get(0, {})
                    perc_class_per_client[0][class_label] = r
                    perc_class_per_client[1] = perc_class_per_client.get(1, {})
                    perc_class_per_client[1][class_label] = 1 - r
                
            else:
                perc_class_per_client = {g: {class_label: 0 for class_label in train_class_counts.keys()} for g in range(G)}
                for class_label in train_class_counts:
                    remaining = 1.0
                    for g in range(G - 1):
                        min_allowed = 0.1
                        max_allowed = remaining - min_allowed * (G - g - 1)
                        r = np.random.uniform(low=min_allowed, high=max_allowed)
                        perc_class_per_client[g] = perc_class_per_client.get(g, {})
                        perc_class_per_client[g][class_label] = r
                        remaining -= r
                    perc_class_per_client[G - 1][class_label] = remaining
                    
    # Case 3: Uniform distribution across clients
    elif perc_class_per_client == 0:
        perc_class_per_client = {g: {class_label: 1/G for class_label in train_class_counts} for g in range(G)}
        
        use_client_sample_props = False
    else:
        use_client_sample_props = True
        
    print(f"Per class per client:{perc_class_per_client}")
    
    class_split = {g: {} for g in range(G)}  

    for class_label, total_available in train_class_counts.items():  
        print(f"Class {class_label}: Total available samples: {total_available}")
        
        if use_client_sample_props and client_sample_props is not None:
            allocations = [
                int(np.floor(
                    total_available *
                    client_sample_props[g] *
                    perc_class_per_client[g].get(class_label, 0)
                ))
                for g in range(G)
            ]
        else:
            allocations = [
                int(np.floor(
                    total_available *
                    perc_class_per_client[g].get(class_label, 0)
                ))
                for g in range(G)
            ]
        
        total_allocated = sum(allocations)
        discrepancy = total_available - total_allocated
        
        print(f"Class {class_label}: Preliminary allocations: {allocations}")
        print(f"Class {class_label}: Total allocated after floor: {total_allocated}")
        print(f"Class {class_label}: Discrepancy: {discrepancy}")
        
        
        # Distribute the discrepancy
        for g in range(G):
            if discrepancy > 0:
                allocations[g] += 1  
                discrepancy -= 1  
                
        if total_available < G:
            # Distribute the available samples evenly among clients 
            for g in range(total_available):  # Distribute one sample to each client until samples run out
                allocations[g] = 1  # Assign one sample to each of the available clients

            # Set remaining clients to 0 if there are fewer samples than clients
            for g in range(total_available, G):
                allocations[g] = 0  
                
        print(f"Class {class_label}: Final allocations per client: {allocations}")   
        
        
        for g in range(G):
            class_split[g][class_label] = allocations[g]  
         
        
    print(f"Class Split:{class_split}")
        
    return class_split

def change_labels(perc_wrong_y, total_samples, num_classes, y, seed=None):
    y = y.copy()
    rng = np.random.default_rng(seed) 

    n_wrong = int(round(perc_wrong_y * total_samples))
    if n_wrong == 0:
        return y

    idx_wrong = rng.choice(total_samples, size=n_wrong, replace=False)

    for i in idx_wrong:
        old = int(y[i])
        new = rng.integers(0, num_classes - 1)
        if new >= old:
            new += 1
        y[i] = new

    return y

    
        
def split_train_test(
    perc_train,
    perc_test,
    perc_wrong_y,
    perc_class_per_client,
    client_sample_props,
    G,
    num_classes, seed, uci_id
):

    # fetch dataset
    all_data = fetch_ucirepo(id=int(uci_id))

    x_all = np.asarray(all_data.data.features)
    y_all = np.asarray(all_data.data.targets)

    if y_all.ndim == 2 and y_all.shape[1] > 1:
        y_all = y_all[:, 1]
    else:
        y_all = y_all.reshape(-1)

    y_all = y_all.flatten()
    
    x_all, y_all = drop_nonfinite_rows(x_all, y_all, name=f"UCI {uci_id} raw", verbose=True)

    # Shuffle
    x_all, y_all = shuffle(x_all, y_all, random_state=42)

    # Encode labels
    y_all, label_mapping = encode_labels(y_all)

    # Sizes / classes
    n_samples, n_features = x_all.shape
    unique_classes, class_counts = np.unique(y_all, return_counts=True)
    unique_classes = np.unique(y_all).astype(int)

    print(f"Class distribution BEFORE split: {dict(zip(unique_classes, class_counts))}")

    # Compute per-class train/test counts
    n_train, n_test = generate_n_sample_vec(y_all, perc_test)

    total_test_samples = sum(n_test.values())
    total_train_samples = sum(n_train.values())

    # Allocate arrays
    x_test = np.zeros((total_test_samples, n_features))
    y_test = np.zeros(total_test_samples)
    x_rem = np.zeros((total_train_samples, n_features))
    y_rem = np.zeros(total_train_samples)

    test_assigned_indices = set()
    train_assigned_indices = set()

    test_index = 0
    rem_index = 0

    # Build x_test and x_rem 
    for c in unique_classes:
        indices_c = np.where(y_all == c)[0]
        x_c = x_all[indices_c]

        # test
        n_test_c = n_test[c]
        test_indices = indices_c[:n_test_c]
        x_test[test_index:test_index + n_test_c] = x_c[:n_test_c]
        y_test[test_index:test_index + n_test_c] = c
        test_assigned_indices.update(test_indices)

        print(f"\nClass {c} - Test Samples:")
        for i, idx in enumerate(test_indices):
            original_label = y_all[idx]
            print(
                f"Original Index: {idx} | Original Label: {original_label} | "
                f"x_test[{test_index + i}]: {x_all[idx]} | y_test[{test_index + i}]: {y_test[test_index + i]}"
            )

        test_index += n_test_c

        # train pool 
        n_train_c = n_train[c]
        train_indices = indices_c[n_test_c:n_test_c + n_train_c]
        x_rem[rem_index:rem_index + n_train_c] = x_c[n_test_c:n_test_c + n_train_c]
        y_rem[rem_index:rem_index + n_train_c] = c
        train_assigned_indices.update(train_indices)

        print(f"\nClass {c} - Train Samples:")
        for i, idx in enumerate(train_indices):
            original_label = y_all[idx]
            print(
                f"Original Index: {idx} | Original Label: {original_label} | "
                f"x_rem[{rem_index + i}]: {x_all[idx]} | y_rem[{rem_index + i}]: {y_rem[rem_index + i]}"
            )

        rem_index += n_train_c

    print("\nFinal Check:")
    print("Total test samples assigned:", len(y_test), "Expected:", sum(n_test.values()))
    print("Total train samples assigned:", len(y_rem), "Expected:", sum(n_train.values()))

    overlap = test_assigned_indices.intersection(train_assigned_indices)
    print("\nOverlap Check:")
    if overlap:
        print("WARNING: Overlapping indices found!", overlap)
    else:
        print("No overlap detected. The split is correct.")

    
    # Fit MinMax scaler on train
    scaler = MinMaxScaler()
    scaler.fit(x_rem)

    x_rem_scaled  = scaler.transform(x_rem)
    x_test_scaled = scaler.transform(x_test)
    x_all_scaled  = scaler.transform(x_all)


    y_test = y_test.astype(int)

    test_hash_before = hash(y_test.tobytes())
    test_counts_before = np.bincount(y_test, minlength=num_classes)

    print("\n[STEP 1] Test set fingerprint BEFORE any label noise:")
    print("  hash(y_test)  =", test_hash_before)
    print("  bincount(y_test) =", test_counts_before)


    unique_classes_rem, counts = np.unique(y_rem, return_counts=True)
    train_class_counts = dict(zip(unique_classes_rem, counts))
    print(f"Train class counts used for splitting:{train_class_counts}")

    class_split = class_split_per_client(
        train_class_counts,
        perc_class_per_client,
        G,
        client_sample_props
    )


    client_sets = {}
    client_sets_norm = {}

    class_used_samples = {c: 0 for c in np.unique(y_rem)}
    allocated_indices = set()
    instance_assignment = {}

    for g in range(G):
        client_sample_count = int(sum(class_split[g].values()))

        
        x_train_g_raw = np.zeros((client_sample_count, n_features))
        y_train_g = np.zeros(client_sample_count)

        
        x_train_g_norm = np.zeros((client_sample_count, n_features))

        prev_idx = 0
        print(f"\nClient {g}: Allocating {client_sample_count} samples.")

        for class_label, sample_count in class_split[g].items():
            class_indices = np.where(y_rem == class_label)[0]
            class_indices = np.sort(class_indices)

            x_c_raw    = x_rem[class_indices, :]          
            x_c_scaled = x_rem_scaled[class_indices, :]   

            start_idx = class_used_samples[class_label]
            end_idx = min(start_idx + sample_count, x_c_raw.shape[0])
            actual_assigned = end_idx - start_idx

            assigned_indices = class_indices[start_idx:end_idx]

            intersection = set(assigned_indices) & allocated_indices
            assert not intersection, f"Duplicate index allocation detected in client {g}: {intersection}"
            allocated_indices.update(assigned_indices)

            # label check
            assert np.all(y_rem[assigned_indices] == class_label), f"Mismatch in labels assigned for client {g}"

            # fill client data
            x_train_g_raw[prev_idx:prev_idx + actual_assigned, :] = x_c_raw[start_idx:end_idx, :]
            y_train_g[prev_idx:prev_idx + actual_assigned] = class_label

            # fill norm client data
            x_train_g_norm[prev_idx:prev_idx + actual_assigned, :] = x_c_scaled[start_idx:end_idx, :]

            for i, orig_idx in enumerate(assigned_indices):
                if prev_idx + i < 10:
                    print(
                        f"Client {g} - Local index {prev_idx + i}: Original Index {orig_idx}, "
                        f"Original Label {y_rem[orig_idx]}, Assigned Label {class_label}"
                    )
                instance_assignment[orig_idx] = (g, prev_idx + i, y_rem[orig_idx])

            print(f"Client {g}: Class {class_label} -> {actual_assigned} samples assigned.")

            prev_idx += actual_assigned
            class_used_samples[class_label] += actual_assigned

        # apply row-wise L2 normalization
        x_train_g_norm = l2_normalize_rows(x_train_g_norm)

        sig_clean = hash(y_train_g.astype(np.int32).tobytes())
        print(f"[STEP 2] Client {g} y signature BEFORE noise: {sig_clean}")

        if perc_wrong_y > 0:
            y_train_g = change_labels(
                perc_wrong_y,
                len(y_train_g),
                num_classes,
                y_train_g,
                seed=seed + 1000 * g
            )

        sig_noisy = hash(y_train_g.astype(np.int32).tobytes())
        print(f"[STEP 2] Client {g} y signature AFTER noise: {sig_noisy}")

        # store raw
        client_sets[f"x{g}"] = x_train_g_raw
        client_sets[f"y{g}"] = y_train_g

        # store norm
        client_sets_norm[f"x{g}"] = x_train_g_norm
        client_sets_norm[f"y{g}"] = y_train_g

        print(f"Unique classes in client {g}: {np.unique(y_train_g)}")

    print("\n=== Instance Assignment Mapping (first 20 entries) ===")
    for idx, mapping in list(instance_assignment.items())[:10]:
        print(f"Original Index {idx} -> Client {mapping[0]}, Local Index {mapping[1]}, Original Label {mapping[2]}")


    central_set = {"x": x_all, "y": y_all}
    test_set = {"x": x_test, "y": y_test}

    x_all_norm = l2_normalize_rows(x_all_scaled)
    x_test_norm = l2_normalize_rows(x_test_scaled)


    central_set_norm = {"x": x_all_norm, "y": y_all}
    test_set_norm = {"x": x_test_norm, "y": y_test}


    test_hash_after = hash(y_test.tobytes())
    test_counts_after = np.bincount(y_test, minlength=num_classes)

    print("\n[STEP 1] Test set fingerprint AT END of split_train_test:")
    print("  hash(y_test)  =", test_hash_after)
    print("  bincount(y_test) =", test_counts_after)

    assert test_hash_after == test_hash_before, "ERROR: y_test changed inside split_train_test!"
    assert np.all(test_counts_after == test_counts_before), "ERROR: y_test class counts changed!"

    return central_set, client_sets, test_set, central_set_norm, client_sets_norm, test_set_norm

In [17]:
class MultiFederatedDRSVM:
    def __init__(self, param, num_clients, num_classes, feature_dim):
        self.epsilon = param['epsilon']
        self.kappa = param['kappa']
        self.pnorm = param['pnorm']
        self.rho = param['rho']  
        self.tau = param['tau'] 
        self.num_clients = num_clients
        self.num_classes = num_classes
        self.feature_dim = feature_dim
        self.alphag_vec = np.ones(self.num_clients)  
        self.Ng_vec = np.zeros(self.num_clients)
        
        self.local_models = [{} for _ in range(num_clients)]
        self.global_model = np.zeros((num_classes, self.feature_dim))  # Weight matrix M: Rows (Classes) * Columns (features)
        #List of mu matrices, one per client
        self.mu = [np.zeros((num_classes, self.feature_dim)) for _ in range(num_clients)] #ADMM dual for each client g: Rows (Classes) * Columns (features)
        
        
    def local_update(self, client_id, x_train, y_train, M_global, mu):

        row, col = x_train.shape
        optimal = {}
        
        encoder = OneHotEncoder(sparse=False)
        y_train_one_hot = encoder.fit_transform(y_train.reshape(-1, 1))
        
        
        # Step 0: Create model
        model = grb.Model(f'DRSVM_Client_{client_id}')
        model.setParam('OutputFlag', False)

        #Step 1: Define decision variables
        var_lambda = model.addVar(vtype=grb.GRB.CONTINUOUS, lb=0.0, name="lambda")
        var_s = var_s = {i: model.addVar(vtype=grb.GRB.CONTINUOUS, lb=0.0, name=f"s[{i}]") for i in range(row)} #Slack variable for each training sample
        
        #Decision variable (Matrix) that holds the weight for class c and feature j
        var_M = {(c, j): model.addVar(vtype=grb.GRB.CONTINUOUS, lb=-grb.GRB.INFINITY)
         for c in range(self.num_classes) for j in range(col)}
        
        # L1 norm constraint
        if self.pnorm == 1:
            slack_var = {(i, j, p): model.addVar(vtype=grb.GRB.CONTINUOUS)
                         for i in range(self.num_classes) 
                         for j in range(self.num_classes) 
                         for p in range(col)}

            
        #Step 2: Integrate variables
        model.update()
        
        #Step 3: Define constraints
        for i in range(len(y_train_one_hot)):
            true_class = np.argmax(y_train_one_hot[i])  
            
            # Iterate over all classes
            for c in range(self.num_classes):
                if c != true_class:  # c is wrong label 

                    # score of the observed label
                    margin_y = grb.quicksum(var_M[(true_class, j)] * x_train[i, j] for j in range(col))

                    # score of label c
                    margin_c = grb.quicksum(var_M[(c, j)] * x_train[i, j] for j in range(col))

                    # CS loss at the observed label true_class 
                    model.addConstr(margin_c - margin_y + 1 <= var_s[i])

                    # CS loss at the label c
                    for j_cls in range(self.num_classes):
                        if j_cls == c:
                            continue

                        margin_j = grb.quicksum(var_M[(j_cls, j)] * x_train[i, j] for j in range(col))
                        model.addConstr(margin_j - margin_c + 1 - self.kappa * var_lambda <= var_s[i])

                    
            
        if self.pnorm == 1:
            for i in range(self.num_classes):
                for j in range(self.num_classes):
                    if i != j:
                        for p in range(col):
                            model.addConstr(var_M[(i, p)] - var_M[(j, p)] <= slack_var[(i, j, p)])
                            model.addConstr(-(var_M[(i, p)] - var_M[(j, p)]) <= slack_var[(i, j, p)])

                        
                        model.addConstr(
                            grb.quicksum(slack_var[(i, j, p)] for p in range(col)) <= var_lambda
                        )


        elif self.pnorm == 2:
            for i in range(self.num_classes):
                for j in range(self.num_classes):
                    if i != j:
                        model.addQConstr(
                            grb.quicksum((var_M[(i, p)] - var_M[(j, p)]) * (var_M[(i, p)] - var_M[(j, p)])
                                         for p in range(col))
                            <= var_lambda * var_lambda
                        )


        elif self.pnorm == float('Inf'):
            for i in range(self.num_classes):
                for j in range(i + 1, self.num_classes):
                    for p in range(col):
                        model.addConstr(var_M[(i, p)] - var_M[(j, p)] <= var_lambda)
                        model.addConstr(-(var_M[(i, p)] - var_M[(j, p)]) <= var_lambda)

        #Step 4: Define objective value
        sum_var_s = grb.quicksum(var_s[i] for i in range(row))
        norm_regularizer = grb.quicksum(
            (var_M[(c, j)] - self.global_model[c, j] + mu[c, j]) * 
            (var_M[(c, j)] - self.global_model[c, j] + mu[c, j])
            for c in range(self.num_classes)
            for j in range(col)
        ) 
        convergence_regularizer = self.tau * grb.quicksum(
            var_M[(c, j)] * var_M[(c, j)]
            for c in range(self.num_classes)
            for j in range(col)
        ) 
        obj = (var_lambda * self.epsilon  
               + ((1 / row) * sum_var_s)  
               + ((self.rho / 2) * norm_regularizer) + convergence_regularizer)

        
        model.setObjective(obj, grb.GRB.MINIMIZE)
        model.optimize()
        
        # Step 6: Store results for the multiclass case
        w_opt = np.zeros((self.num_classes, col))
   
        for c in range(self.num_classes):
            for j in range(col):
                w_opt[c, j] = var_M[(c, j)].x

        tmp = {'w': w_opt, 'objective': model.ObjVal, 'diagnosis': model.status}
        optimal.update(tmp)

        return optimal
    
    def train(self, client_sets, num_rounds):
       
        self.Ng_vec = np.array([len(client_sets['x' + str(i)]) for i in range(self.num_clients)])

        prev_global_model = np.copy(self.global_model)
        
        for num_round in range(num_rounds):
            local_models = [] #Store local weights for each client
            
            for i in range(self.num_clients):
                
                x_train = client_sets['x' + str(i)]
                y_train = client_sets['y' + str(i)]
                
                #Solve each clients local problem
                local_M = self.local_update(i, x_train, y_train, self.global_model, self.mu[i])
                #Store clients M_g
                local_models.append(local_M)
            
            self.alphag_vec = self.Ng_vec / np.sum(self.Ng_vec)
            
            local_updates = np.array([local_models[i]['w'] + self.mu[i] for i in range(self.num_clients)])

            weighted_updates = self.alphag_vec[:, np.newaxis, np.newaxis] * local_updates  

            self.global_model = np.sum(weighted_updates, axis=0)  

            diff_weights = np.linalg.norm(self.global_model - prev_global_model)
            print(f"Weight change norm (round {num_round}):", diff_weights)
            
            prev_global_model = np.copy(self.global_model)
            
            for i in range(self.num_clients):
                self.mu[i] += local_models[i]['w'] - self.global_model  
        

        return self.global_model


    def test(self, client_sets_norm):
        """Test the model with provided data (N samples, P features)"""

        x_test = client_sets_norm['x']
        row_x, col_x = x_test.shape
        C = self.global_model.shape[0]  # Number of classes
        y_pred = np.zeros((row_x, C), dtype=int)  # One-hot encoded output

        w_global = self.global_model  # Access the trained weights from the model

        for n in range(row_x):
            # Compute the linear output for each sample
            linear_output = np.matmul(w_global, x_test[n])
            
            # Get the class with the maximum score
            class_index = np.argmax(linear_output)
            
            y_pred[n, class_index] = 1


        return y_pred

In [18]:
class MultiFederatedPrivateDRSVM_Linearized:

    def __init__(self, param, num_clients, num_classes):
        # DRO + ADMM params
        self.epsilon = float(param["epsilon"])
        self.kappa   = float(param["kappa"])
        self.pnorm   = param["pnorm"]   
        self.rho     = float(param["rho"])
        # Bertsekas penalty parameters
        self.gamma_pen = float(param.get("gamma_pen", 1.0))   
        self.tau_init  = float(param.get("tau_init", 0.0))    
        self.nu      = param.get("nu", None)  

        self.num_clients = int(num_clients)
        self.num_classes = int(num_classes)

        # DP params (optional)
        self.dp_on    = bool(param.get("dp_on", False))
        self.eps_dp   = param.get("eps_dp", None)
        self.delta_dp = param.get("delta_dp", None)
        self.cw       = float(param.get("cw", 1.0))
        self.L = float(param.get("L", 1.0))   # loss M-block bound
        self.C = float(param.get("C", 1.0))   # penalty M-block bound

        self.feature_dim  = None
        self.Ng_vec       = None
        self.alphag_vec   = None

        self.global_model = None               
        self.mu           = None              

        self.M_prev       = None               
        self.lam_prev     = None               
        self.tau_prev = None
        
        self.clambda = float(param.get("clambda", 1.0))
        self.S_lam = float(param.get("S_lam", 1.0))
        self.V_lam = float(param.get("V_lam", 1.0))

    # DP step sizes
    def eta_gk(self, g, k):
        Ng = float(self.Ng_vec[g])
        d  = int(self.feature_dim)
        K  = int(self.num_classes)
        dp = d * K

        term = (
            2.0 * (self.L + self.C)**2
            + dp * (8.0 * self.L**2 / (Ng**2)) * (2.0 * np.log(1.25 / self.delta_dp) / (self.eps_dp**2))
        )
        return self.cw / np.sqrt(2.0 * k * term)

    def nu_gk(self, k: int) -> float:

        c_lam = float(self.clambda if self.clambda is not None else 1.0)
        denom = np.sqrt(2.0 * k) * (self.S_lam + self.V_lam)
        if c_lam <= 0:
            raise ValueError("clambda must be > 0")
        if denom <= 0:
            raise ValueError("Need S_lam + V_lam > 0 to define nu_gk")
        return c_lam / denom
    

    def sigma_gk(self, g, k):
        Ng  = float(self.Ng_vec[g])
        eta = self.eta_gk(g, k)
        return ((2.0*np.sqrt(2.0)*self.L) / (Ng*(self.rho + 1.0/eta))
                * np.sqrt(2.0*np.log(1.25/self.delta_dp)) / self.eps_dp)
    
    # Past iteration function evaluation 
    def _cs_hinge_value(self, scores, y):

        tmp = scores.copy()
        tmp[y] = -np.inf
        c_star = int(np.argmax(tmp))
        H = 1.0 + scores[c_star] - scores[y]
        return float(max(0.0, H))


    def phi_value(self, M):

        K, d = M.shape
        best_val = -np.inf

        for i in range(K):
            for j in range(K):
                if i == j:
                    continue
                u = M[i, :] - M[j, :]

                if self.pnorm == 1:
                    val = np.sum(np.abs(u))              # ||u||_1
                elif self.pnorm == 2:
                    val = np.linalg.norm(u, 2)           # ||u||_2
                elif self.pnorm == np.inf or self.pnorm == float('Inf'):
                    val = np.max(np.abs(u))              # ||u||_inf
                else:
                    raise ValueError("Use pnorm=1, 2, or np.inf.")

                if val > best_val:
                    best_val = val

        return float(best_val)


    def bertsekas_penalty_value(self, M, lam, gamma_pen, tau):

        phi = self.phi_value(M)
        r = float(tau + gamma_pen * (phi - lam))
        return (max(0.0, r)**2 - float(tau)**2) / (2.0 * float(gamma_pen))


    def f_value(self, M, lam, x_train, y_train, gamma_pen, tau):

        K = self.num_classes
        N, d = x_train.shape
        if N == 0:
            return 0.0

        y_train = y_train.astype(int)
        total = 0.0

        for n in range(N):
            x = x_train[n]
            y_true = int(y_train[n])
            scores = M @ x

            A = self._cs_hinge_value(scores, y_true)

            B_best = -np.inf
            for y in range(K):
                if y == y_true:
                    continue
                loss_y = self._cs_hinge_value(scores, y)
                val = loss_y - self.kappa * lam
                if val > B_best:
                    B_best = val

            total += max(A, B_best)

        loss_part = total / float(N)
        pen_part = self.bertsekas_penalty_value(M, lam, gamma_pen, tau)

        return float(loss_part + pen_part)

    
    # compute largest dual between any two class weights
    def phi_and_subgrad(self, M):

        K, d = M.shape
        best_val = -np.inf
        best_i = best_j = None
        best_u = None

        for i in range(K):
            for j in range(K):
                if i == j:
                    continue

                u = M[i, :] - M[j, :]

                if self.pnorm == 1:
                    val = np.sum(np.abs(u))
                elif self.pnorm == 2:
                    val = np.linalg.norm(u, 2)
                elif self.pnorm == np.inf or self.pnorm == float("Inf"):
                    val = np.max(np.abs(u))
                else:
                    raise ValueError("Use pnorm=1, 2, or np.inf.")

                if val > best_val:
                    best_val = val
                    best_i, best_j = i, j
                    best_u = u.copy()

        phi = float(best_val)
        Gphi = np.zeros_like(M)

        if best_u is None:
            return 0.0, Gphi

        if self.pnorm == 1:
            
            g = np.sign(best_u)   

        elif self.pnorm == 2:
            norm_u = np.linalg.norm(best_u, 2)
            if norm_u > 1e-12:
                g = best_u / norm_u
            else:
                g = np.zeros(d)

        else:  
            p_star = int(np.argmax(np.abs(best_u)))
            g = np.zeros(d)
            if abs(best_u[p_star]) > 1e-12:
                g[p_star] = np.sign(best_u[p_star])
            else:
                g[p_star] = 0.0

        Gphi[best_i, :] += g
        Gphi[best_j, :] -= g

        return phi, Gphi

    #  subgradient at previous iterate 
    def _cs_hinge_and_grad(self, scores, x, y):

        K = scores.shape[0]
        # best competitor class 
        tmp = scores.copy()
        tmp[y] = -np.inf
        c_star = int(np.argmax(tmp))

        H = 1.0 + scores[c_star] - scores[y]
        if H <= 0.0:
            return 0.0, None  

        # subgrad wrt M
        return float(H), (c_star, y)  


    def subgrad_f(self, M_prev, lam_prev, x_train, y_train, gamma_pen, tau):

        K = self.num_classes
        N, d = x_train.shape
        if N == 0:
            return np.zeros((K, d), dtype=float), 0.0

        y_train = y_train.astype(int)

        G_M = np.zeros((K, d), dtype=float)
        g_lam = 0.0

        # loss part
        for n in range(N):
            x = x_train[n]
            y_true = int(y_train[n])

            scores = M_prev @ x  
            
            A_loss, A_pair = self._cs_hinge_and_grad(scores, x, y_true)

            B_best = -np.inf
            B_loss = 0.0
            B_pair = None

            for y in range(K):
                if y == y_true:
                    continue
                loss_y, pair_y = self._cs_hinge_and_grad(scores, x, y)
                val = loss_y - self.kappa * lam_prev
                if val > B_best:
                    B_best = val
                    B_loss = loss_y
                    B_pair = pair_y

            # outer max
            if A_loss >= B_best:
                # choose A branch
                if A_pair is not None:
                    c_star, y = A_pair
                    G_M[c_star, :] += x
                    G_M[y,      :] -= x
                # no lambda term from A
            else:
                # choose B branch
                if B_pair is not None:
                    c_star, y = B_pair
                    G_M[c_star, :] += x
                    G_M[y,      :] -= x
                # lambda derivative
                g_lam += -self.kappa

        # average loss subgrad 
        G_M /= float(N)
        g_lam /= float(N)

        # Bertsekas penalty part 
        phi_val, G_phi = self.phi_and_subgrad(M_prev)
        r = float(tau + gamma_pen * (phi_val - lam_prev))
        if r > 0.0:
            G_M += r * G_phi
            g_lam += -r
        #returns subgradient wrt M and wrt lambda    
        return G_M, g_lam


    #  linearized local QP solve 
    def local_update_linearized(self, client_id, x_train, y_train, M_global, mu, k):

        N, d = x_train.shape
        K = self.num_classes
        

        M_prev  = self.M_prev[client_id]
        lam_prev = float(self.lam_prev[client_id])

        if self.dp_on:
            eta = self.eta_gk(client_id, k)
        else:
            eta = self.cw / np.sqrt(2.0 * max(k, 1))

        if self.nu is None:
            nu = self.nu_gk(k)
        else:
            nu = float(self.nu)

        tau = float(self.tau_prev[client_id])
        gamma_pen = float(self.gamma_pen)

        G_M, g_lam = self.subgrad_f(
            M_prev, lam_prev, x_train, y_train,
            gamma_pen=gamma_pen, tau=tau
        )

        f_prev = self.f_value(
            M_prev, lam_prev, x_train, y_train,
            gamma_pen=gamma_pen, tau=tau
        )

        model = grb.Model(f"LIN_DRSVM_Client_{client_id}")
        model.setParam("OutputFlag", False)

        var_lambda = model.addVar(vtype=grb.GRB.CONTINUOUS, lb=0.0)
        var_M = {(c, j): model.addVar(vtype=grb.GRB.CONTINUOUS, lb=-grb.GRB.INFINITY)
                 for c in range(K) for j in range(d)}

        model.update()

        # Objective
        obj = 0.0

        # lambda * epsilon
        obj += var_lambda * self.epsilon

        # <G_M, M - M_prev>
        obj += grb.quicksum(
            float(G_M[c, j]) * (var_M[(c, j)] - float(M_prev[c, j]))
            for c in range(K) for j in range(d)
        )

        # g_lam * (lambda - lam_prev)
        obj += float(g_lam) * (var_lambda - float(lam_prev))

        # rho/2 ||M - M_global + mu||^2
        obj += (self.rho / 2.0) * grb.quicksum(
            (var_M[(c, j)] - float(M_global[c, j]) + float(mu[c, j]))
            * (var_M[(c, j)] - float(M_global[c, j]) + float(mu[c, j]))
            for c in range(K) for j in range(d)
        )

        # 1/(2 eta) ||M - M_prev||^2
        obj += (1.0 / (2.0 * eta)) * grb.quicksum(
            (var_M[(c, j)] - float(M_prev[c, j])) * (var_M[(c, j)] - float(M_prev[c, j]))
            for c in range(K) for j in range(d)
        )

        # 1/(2 nu) (lambda - lam_prev)^2
        obj += (1.0 / (2.0 * nu)) * (var_lambda - float(lam_prev)) * (var_lambda - float(lam_prev))
        

        model.setObjective(obj, grb.GRB.MINIMIZE)
        model.optimize()

        status = model.Status
        print(f"[client {client_id}, round {k}] status = {status}")

        if status != grb.GRB.OPTIMAL:
            print(f"[client {client_id}, round {k}] non-optimal solve")
            print(f"  eta = {eta}")
            print(f"  nu = {nu}")
            print(f"  tau = {tau}")
            print(f"  gamma_pen = {gamma_pen}")
            print(f"  lam_prev = {lam_prev}")
            print(f"  g_lam = {g_lam}")
            print(f"  f_prev = {f_prev}")
            print(f"  any NaN in G_M? {np.isnan(G_M).any()}")
            print(f"  any Inf in G_M? {np.isinf(G_M).any()}")
            print(f"  any NaN in M_prev? {np.isnan(M_prev).any()}")
            print(f"  any Inf in M_prev? {np.isinf(M_prev).any()}")

            return {
                "w": M_prev.copy(),
                "lambda": float(lam_prev),
                "objective_qp": np.nan,
                "f_prev": float(f_prev),
                "Lhat": np.nan,
                "status": int(status),
                "eta_used": float(eta),
            }

        obj_val = float(model.ObjVal)
        Lhat = float(f_prev + obj_val)

        w_opt = np.zeros((K, d), dtype=float)
        for c in range(K):
            for j in range(d):
                w_opt[c, j] = var_M[(c, j)].X

        return {
            "w": w_opt,
            "lambda": float(var_lambda.X),
            "objective_qp": obj_val,
            "f_prev": float(f_prev),
            "Lhat": Lhat,
            "status": int(status),
            "eta_used": float(eta),
        }

    #  training loop 
    def train(self, client_sets, num_rounds):
        # sizes
        self.Ng_vec = np.array([len(client_sets[f"x{i}"]) for i in range(self.num_clients)], dtype=float)
        self.alphag_vec = self.Ng_vec / np.sum(self.Ng_vec)

        # init dims + vars
        if self.feature_dim is None:
            self.feature_dim = int(client_sets["x0"].shape[1])

        K, d = self.num_classes, self.feature_dim

        if self.global_model is None:
            self.global_model = np.zeros((K, d), dtype=float)
        if self.mu is None:
            self.mu = [np.zeros((K, d), dtype=float) for _ in range(self.num_clients)]

        if self.M_prev is None:
            self.M_prev = [np.zeros((K, d), dtype=float) for _ in range(self.num_clients)]
        if self.lam_prev is None:
            self.lam_prev = [0.0 for _ in range(self.num_clients)]
        if self.tau_prev is None:
            self.tau_prev = [float(self.tau_init) for _ in range(self.num_clients)]

        # L setup 
        if self.dp_on:
            if self.L is None:
                self.L = 1
            if self.eps_dp is None or self.delta_dp is None:
                raise ValueError("dp_on=True requires eps_dp and delta_dp")
            if self.eps_dp <= 0:
                raise ValueError("eps_dp must be > 0")
            if not (0 < self.delta_dp < 1):
                raise ValueError("delta_dp must be in (0,1)")

        prev_global = self.global_model.copy()

        for k in range(1, int(num_rounds) + 1):
            local_models = []

            for i in range(self.num_clients):
                x_i = client_sets[f"x{i}"]
                y_i = client_sets[f"y{i}"].astype(int)

                out = self.local_update_linearized(
                    client_id=i,
                    x_train=x_i,
                    y_train=y_i,
                    M_global=self.global_model,
                    mu=self.mu[i],
                    k=k
                )

                w_clean = out["w"]
                lam_clean = out["lambda"]

                
                self.M_prev[i] = w_clean.copy()
                self.lam_prev[i] = float(lam_clean)
                
                #  alpha update 
                phi_val, _ = self.phi_and_subgrad(w_clean)
                gamma_pen = float(self.gamma_pen)
                self.tau_prev[i] = max(0.0, self.tau_prev[i] + gamma_pen * (phi_val - lam_clean))
                
                # DP noise
                if self.dp_on:
                    sigma = self.sigma_gk(i, k)
                    noise = np.random.normal(0.0, sigma, size=w_clean.shape)
                    w_send = w_clean + noise
                else:
                    w_send = w_clean

                out["w_clean"] = w_clean
                out["w_send"] = w_send
                local_models.append(out)

            
            local_updates = np.array([local_models[i]["w_send"] + self.mu[i] for i in range(self.num_clients)])
            weighted = self.alphag_vec[:, None, None] * local_updates
            self.global_model = np.sum(weighted, axis=0)

            # dual update
            for i in range(self.num_clients):
                self.mu[i] += local_models[i]["w_send"] - self.global_model

            diff = np.linalg.norm(self.global_model - prev_global)
            print(f"[LIN-DRO] round {k}: ||w^k - w^{k-1}||_F = {diff:.4e}")
            prev_global = self.global_model.copy()

        return self.global_model

    # ---------------- prediction ----------------
    def test(self, test_set):
        x_test = test_set["x"]
        scores = x_test @ self.global_model.T
        y_hat = np.argmax(scores, axis=1)
        
        counts = np.bincount(y_hat, minlength=self.num_classes)
        print("Pred counts:", counts)

        N = x_test.shape[0]
        K = self.num_classes
        y_pred = np.zeros((N, K), dtype=int)
        y_pred[np.arange(N), y_hat] = 1
        return y_pred

In [19]:
class MultiFederatedERMSVM_DP_ADMM:


    def __init__(self, param, num_clients, num_classes):
        self.num_clients = num_clients
        self.num_classes = num_classes

        # ADMM / ERM params
        self.rho = param.get("rho", 1.0)
        self.lambda_reg = param.get("lambda_reg", 1e-2)  

        self.dp_on    = param.get("dp_on", False)
        self.eps_dp   = param.get("eps_dp", None)
        self.delta_dp = param.get("delta_dp", None)
        self.cw       = param.get("cw", 1.0)
        self.L        = param.get("L", None)  

        self.feature_dim = None
        self.Ng_vec = None

        self.global_model = None            
        self.w_tilde_prev = None            
        self.gamma = None                   



    #  DP step sizes
    def eta_gk(self, i, k):
        Ng = self.Ng_vec[i]
        d  = self.feature_dim
        p  = self.num_classes
        dp = d * p

        term = self.L**2 + dp * (4*self.L**2/(Ng**2)) * (2*np.log(1.25/self.delta_dp)/(self.eps_dp**2))
        return self.cw / np.sqrt(2.0 * k * term)

    def sigma_gk(self, i, k):
        Ng = self.Ng_vec[i]
        eta = self.eta_gk(i, k)
        return (2.0*np.sqrt(2.0)*self.L)/(Ng*(self.rho + 1.0/eta)) * np.sqrt(2.0*np.log(1.25/self.delta_dp))/self.eps_dp

    #  ERM loss subgradient at M_prev 
    def subgrad_multiclass_hinge(self, M_prev, x, y):

        K, d = M_prev.shape
        scores = M_prev @ x
        y = int(y)

        tmp = scores.copy()
        tmp[y] = -np.inf
        c_star = int(np.argmax(tmp))

        margin = 1.0 + scores[c_star] - scores[y]
        if margin <= 0.0:
            return np.zeros((K, d), dtype=float)

        G = np.zeros((K, d), dtype=float)
        G[c_star, :] += x
        G[y,      :] -= x
        return G

    def grad_f_i(self, i, x_i, y_i, M_prev):

        mi = x_i.shape[0]
        K, d = M_prev.shape

        if mi == 0:
            return np.zeros((K, d), dtype=float)

        G = np.zeros((K, d), dtype=float)
        for j in range(mi):
            G += self.subgrad_multiclass_hinge(M_prev, x_i[j], y_i[j])
        G /= float(mi)

        # regularizer
        G += (self.lambda_reg / float(self.num_clients)) * M_prev
        return G

    # closed-form solution of the prox-linearized subproblem
    def local_step_closed_form(self, i, k, x_i, y_i):

        w_prev = self.global_model
        wtilde_prev = self.w_tilde_prev[i]
        gamma_prev = self.gamma[i]

        eta = self.eta_gk(i, k)
        g = self.grad_f_i(i, x_i, y_i, wtilde_prev)

        denom = (self.rho + 1.0/eta)
        w_clean = (self.rho * w_prev + (1.0/eta) * wtilde_prev + gamma_prev - g) / denom
        return w_clean

    def train(self, client_sets, num_rounds):
        
        self.feature_dim = client_sets["x0"].shape[1]
        K, d = self.num_classes, self.feature_dim

        self.Ng_vec = np.array([len(client_sets[f"x{i}"]) for i in range(self.num_clients)])

        if self.L is None:
            self.L = np.sqrt(self.feature_dim)

        # DP 
        if self.dp_on:
            if self.eps_dp is None or self.delta_dp is None:
                raise ValueError("dp_on=True requires eps_dp and delta_dp")
            if self.eps_dp <= 0:
                raise ValueError("eps_dp must be > 0")
            if not (0 < self.delta_dp < 1):
                raise ValueError("delta_dp must be in (0,1)")

        if self.global_model is None:
            self.global_model = np.zeros((K, d), dtype=float)         
        if self.gamma is None:
            self.gamma = [np.zeros((K, d), dtype=float) for _ in range(self.num_clients)]  
        if self.w_tilde_prev is None:
            self.w_tilde_prev = [np.zeros((K, d), dtype=float) for _ in range(self.num_clients)] 

        for k in range(1, num_rounds + 1):
            w_clean_list = []
            w_tilde_list = []

            for i in range(self.num_clients):
                x_i = client_sets[f"x{i}"]
                y_i = client_sets[f"y{i}"].astype(int)

                w_clean = self.local_step_closed_form(i, k, x_i, y_i)

                if self.dp_on:
                    sigma = self.sigma_gk(i, k)
                    noise = np.random.normal(0.0, sigma, size=w_clean.shape)
                    w_tilde = w_clean + noise
                else:
                    w_tilde = w_clean

                w_clean_list.append(w_clean)
                w_tilde_list.append(w_tilde)

            # server update
            w_tilde_bar = np.mean(np.array(w_tilde_list), axis=0)
            gamma_bar = np.mean(np.array(self.gamma), axis=0)

            self.global_model = w_tilde_bar - (gamma_bar / self.rho)

            # dual update
            for i in range(self.num_clients):
                self.gamma[i] = self.gamma[i] - self.rho * (w_tilde_list[i] - self.global_model)
                self.w_tilde_prev[i] = w_tilde_list[i].copy()

        return self.global_model

    def test(self, test_set):
        x_test = test_set["x"]
        scores = x_test @ self.global_model.T
        y_hat = np.argmax(scores, axis=1)

        N = x_test.shape[0]
        K = self.num_classes
        y_pred = np.zeros((N, K), dtype=int)
        y_pred[np.arange(N), y_hat] = 1
        return y_pred

In [20]:
def _to_label_vector(y):
    y = np.asarray(y)
    if y.ndim > 1:
        y = y.reshape(-1)
    return y.astype(int)

def macro_f1_score(y_true, y_pred, num_classes):

    y_true = _to_label_vector(y_true)
    y_pred = _to_label_vector(y_pred)

    f1s = []
    for c in range(num_classes):
        tp = np.sum((y_true == c) & (y_pred == c))
        fp = np.sum((y_true != c) & (y_pred == c))
        fn = np.sum((y_true == c) & (y_pred != c))

        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0

        f1 = (2 * prec * rec / (prec + rec)) if (prec + rec) > 0 else 0.0
        f1s.append(f1)

    return float(np.mean(f1s))

def predict_labels(model, test_set, num_classes):
    out = np.asarray(model.test(test_set))
    if out.ndim == 1:
        return out.astype(int)
    if out.ndim == 2 and out.shape[1] == num_classes:
        return np.argmax(out, axis=1).astype(int)
    raise ValueError(f"Unrecognized prediction output shape: {out.shape}")


In [21]:
# 3) Dataset split caching
class SplitCache:

    def __init__(self):
        self._cache = {}

    def get(self, dataset_name, seed, split_fn):
        key = (dataset_name, int(seed))
        if key not in self._cache:
            np.random.seed(int(seed))
            self._cache[key] = split_fn(int(seed))
        return self._cache[key]


_split_cache = SplitCache()


# 4) Dataset config helper
def make_dataset_config(name, split_fn_seeded, num_classes):
    return {
        "name": str(name),
        "num_classes": int(num_classes),
        "get_split": split_fn_seeded,   
    }

In [22]:
# ---------------- labels: 0 normal, 1 leak, 2 blocking, 3 bearing ----------------
def build_fault_labels_numeric_from_flags(df):
    """
    Classes:
      0 = normal (no flags)
      1 = LeakFlag == 1
      2 = BlockingFlag == 1
      3 = BearingFlag == 1
    """
    df = df.copy()
    df.columns = df.columns.str.strip()

    leak  = df["LeakFlag"].astype(int).to_numpy()
    block = df["BlockingFlag"].astype(int).to_numpy()
    bear  = df["BearingFlag"].astype(int).to_numpy()

    s = leak + block + bear
    bad = np.where(s > 1)[0]
    if len(bad) > 0:
        raise ValueError(f"Multiple flags ON in {len(bad)} rows. Example row index {bad[0]}")

    y = np.zeros(len(df), dtype=int)
    y[leak == 1]  = 1
    y[block == 1] = 2
    y[bear == 1]  = 3
    return y

def split_train_test_csv(
    csv_path,
    perc_train,
    perc_test,
    perc_wrong_y,
    perc_class_per_client,
    client_sample_props,
    G,
    num_classes,
    label_col=None,
    label_builder=None,
    feature_cols=None,
    drop_cols=None,
    feature_scaling="minmax",
    shuffle_seed=42
):

    # 1) load csv
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()

    # 2) drop rows with missing values
    df = df.dropna().copy()

    # 3) build labels
    if label_builder is not None:
        y_all = label_builder(df)
    elif label_col is not None:
        y_all = df[label_col].to_numpy()
        y_all, label_mapping = encode_labels(y_all)
    else:
        raise ValueError("Provide either label_col or label_builder.")

    # 4) build features
    if feature_cols is None:
        cols_to_drop = [] if drop_cols is None else list(drop_cols)
        if label_col is not None:
            cols_to_drop.append(label_col)
        X_df = df.drop(columns=cols_to_drop, errors="ignore")
    else:
        X_df = df[feature_cols].copy()
        
    # 5) one-hot encode categorical columns automatically
    X_df = pd.get_dummies(X_df, drop_first=False)

    x_all = X_df.to_numpy(dtype=float)

    x_all, y_all = drop_nonfinite_rows(x_all, y_all, name=f"CSV {csv_path}", verbose=True)

    # 6) shuffle
    x_all, y_all = shuffle(x_all, y_all, random_state=shuffle_seed)

    # sanity
    if y_all.min() < 0 or y_all.max() >= num_classes:
        raise ValueError(
            f"Label range mismatch. min={y_all.min()} max={y_all.max()} num_classes={num_classes}"
        )

    # 8) class distribution before split
    unique_classes, class_counts = np.unique(y_all, return_counts=True)
    print(f"Class distribution BEFORE split: {dict(zip(unique_classes, class_counts))}")

    # 9) compute train/test counts
    n_train, n_test = generate_n_sample_vec(y_all, perc_test)

    n_samples, n_features = x_all.shape
    total_test_samples  = sum(n_test.values())
    total_train_samples = sum(n_train.values())

    x_test = np.zeros((total_test_samples, n_features))
    y_test = np.zeros(total_test_samples, dtype=int)
    x_rem  = np.zeros((total_train_samples, n_features))
    y_rem  = np.zeros(total_train_samples, dtype=int)

    test_index = 0
    rem_index  = 0

    # 10) per-class split
    for c in np.unique(y_all).astype(int):
        indices_c = np.where(y_all == c)[0]
        x_c = x_all[indices_c]

        n_test_c = n_test[c]
        x_test[test_index:test_index + n_test_c] = x_c[:n_test_c]
        y_test[test_index:test_index + n_test_c] = c
        test_index += n_test_c

        n_train_c = n_train[c]
        x_rem[rem_index:rem_index + n_train_c] = x_c[n_test_c:n_test_c + n_train_c]
        y_rem[rem_index:rem_index + n_train_c] = c
        rem_index += n_train_c

    # 11) feature scaling fit on training only
    scaler = None
    if feature_scaling == "minmax":
        scaler = MinMaxScaler()
        x_rem_scaled  = scaler.fit_transform(x_rem)
        x_test_scaled = scaler.transform(x_test)
    elif feature_scaling is None:
        x_rem_scaled, x_test_scaled = x_rem, x_test
    else:
        raise ValueError('feature_scaling must be "minmax" or None')

    # 12) row-wise normalization
    x_rem_proc  = l2_normalize_rows(x_rem_scaled)
    x_test_proc = l2_normalize_rows(x_test_scaled)

    # 13) class counts for client splitting
    unique_classes_rem, counts = np.unique(y_rem, return_counts=True)
    train_class_counts = dict(zip(unique_classes_rem, counts))
    print(f"Train class counts used for splitting: {train_class_counts}")

    class_split = class_split_per_client(train_class_counts, perc_class_per_client, G, client_sample_props)

    client_sets = {}
    client_sets_norm = {}

    class_used_samples = {c: 0 for c in np.unique(y_rem)}

    for g in range(G):
        client_sample_count = int(sum(class_split[g].values()))
        x_train_g = np.zeros((client_sample_count, n_features))
        y_train_g = np.zeros(client_sample_count, dtype=int)

        prev_idx = 0

        for class_label, sample_count in class_split[g].items():
            x_c = x_rem_proc[y_rem == class_label, :]
            start_idx = class_used_samples[class_label]
            end_idx = min(start_idx + sample_count, x_c.shape[0])
            actual = end_idx - start_idx

            x_train_g[prev_idx:prev_idx + actual, :] = x_c[start_idx:end_idx, :]
            y_train_g[prev_idx:prev_idx + actual] = class_label

            prev_idx += actual
            class_used_samples[class_label] += actual

        if perc_wrong_y > 0:
            y_train_g = change_labels(
                perc_wrong_y=perc_wrong_y,
                total_samples=len(y_train_g),
                num_classes=num_classes,
                y=y_train_g
            )

        client_sets[f"x{g}"] = x_train_g
        client_sets[f"y{g}"] = y_train_g

        client_sets_norm[f"x{g}"] = x_train_g
        client_sets_norm[f"y{g}"] = y_train_g

    central_set = {"x": x_all, "y": y_all}

    if scaler is not None:
        x_all_scaled = scaler.transform(x_all)
    else:
        x_all_scaled = x_all

    central_set_norm = {"x": l2_normalize_rows(x_all_scaled), "y": y_all}

    test_set = {"x": x_test, "y": y_test}
    test_set_norm = {"x": x_test_proc, "y": y_test}

    return central_set, client_sets, test_set, central_set_norm, client_sets_norm, test_set_norm, X_df.columns.tolist()

In [24]:
def make_get_split_real(
    uci_id,
    num_classes,
    perc_train,
    perc_test,
    perc_wrong_y,
    perc_class_per_client,
    client_sample_props,
    G,
):
    def _get_split(seed: int):
        
        (_, _, _,
         _, client_sets_norm, test_set_norm) = split_train_test(
            uci_id=int(uci_id),
            perc_train=float(perc_train),
            perc_test=float(perc_test),
            perc_wrong_y=float(perc_wrong_y),
            perc_class_per_client=perc_class_per_client,
            client_sample_props=client_sample_props,
            G=int(G),
            num_classes=int(num_classes),
            seed=int(seed), 
        )
        return client_sets_norm, test_set_norm
    return _get_split


def make_get_split_csv(
    csv_path,
    num_classes,
    perc_train,
    perc_test,
    perc_wrong_y,
    perc_class_per_client,
    client_sample_props,
    G,
    label_col=None,
    label_builder=None,
    feature_scaling="minmax",
    feature_cols=None,
    drop_cols=None,
):
    feat_cols = feature_cols
    dcols = drop_cols

    def _get_split(seed: int):
        (
            central_set, client_sets, test_set,
            central_set_norm, client_sets_norm, test_set_norm, used_feature_cols
        ) = split_train_test_csv(
            csv_path=csv_path,
            perc_train=perc_train,
            perc_test=perc_test,
            perc_wrong_y=perc_wrong_y,
            perc_class_per_client=perc_class_per_client,
            client_sample_props=client_sample_props,
            G=G,
            num_classes=num_classes,
            label_col=label_col,
            label_builder=label_builder,
            feature_cols=feat_cols,
            drop_cols=dcols,
            shuffle_seed=int(seed),
            feature_scaling=feature_scaling,
        )
        return client_sets_norm, test_set_norm

    return _get_split

In [25]:
# 5) Model training wrappers 


tau = 0.0
TAU = 0.0

def train_model_dp_erm(
    client_norm,
    num_classes,
    seed,
    eps_dp,
    delta_dp,
    num_rounds,
    rho_erm=None,
    cw=None,
    L_lip=None,
    lambda_reg=None,
    G=None,
):
    if rho_erm is None:
        rho_erm = RHO_ERM
    if cw is None:
        cw = CW
    if L_lip is None:
        L_lip = L_LIP
    if lambda_reg is None:
        lambda_reg = LAMBDA_REG
    if G is None:
        G = G_CONST


    param = {
        "rho": float(rho_erm),
        "lambda_reg": float(lambda_reg),
        "dp_on": True,
        "eps_dp": float(eps_dp),
        "delta_dp": float(delta_dp),
        "cw": float(cw),
        "L": float(L_lip),
    }

    model = MultiFederatedERMSVM_DP_ADMM(
        param,
        num_clients=int(G),
        num_classes=int(num_classes),
    )
    model.train(client_norm, num_rounds=int(num_rounds))
    return model


def train_model_dp_dro_linearized(
    client_norm,
    num_classes,
    seed,
    eps_dp,
    delta_dp,
    num_rounds,
    dro_ambig_eps=None,
    kappa=None,
    pnorm=None,
    rho_dro=None,
    cw=None,
    L_lip=None,
    C_pen=None,
    gamma_pen=None,
    tau_init=None,
    clambda=None,
    S_lam=None,
    V_lam=None,
    G=None,
):
    if dro_ambig_eps is None:
        dro_ambig_eps = DRO_AMBIG_EPS
    if kappa is None:
        kappa = KAPPA
    if pnorm is None:
        pnorm = PNORM
    if rho_dro is None:
        rho_dro = RHO_DRO
    if cw is None:
        cw = CW
    if L_lip is None:
        L_lip = L_LIP
    if C_pen is None:
        C_pen = C_PEN
    if gamma_pen is None:
        gamma_pen = GAMMA_PEN
    if tau_init is None:
        tau_init = TAU_INIT
    if clambda is None:
        clambda = C_LAMBDA
    if S_lam is None:
        S_lam = S_LAM
    if V_lam is None:
        V_lam = V_LAM
    if G is None:
        G = G_CONST


    param = {
        "epsilon": float(dro_ambig_eps),
        "kappa": float(kappa),
        "pnorm": pnorm,
        "rho": float(rho_dro),

        "gamma_pen": float(gamma_pen),
        "tau_init": float(tau_init),

        "dp_on": True,
        "eps_dp": float(eps_dp),
        "delta_dp": float(delta_dp),

        "cw": float(cw),
        "L": float(L_lip),
        "C": float(C_pen),

        "clambda": float(clambda),
        "S_lam": float(S_lam),
        "V_lam": float(V_lam),
    }

    model = MultiFederatedPrivateDRSVM_Linearized(
        param,
        num_clients=int(G),
        num_classes=int(num_classes),
    )
    model.train(client_norm, num_rounds=int(num_rounds))
    return model


def train_model_np_dro(
    client_norm,
    num_classes,
    seed,
    num_rounds,
    dro_ambig_eps=None,
    kappa=None,
    pnorm=None,
    rho_dro=None,
    tau=None,
    G=None,
):
    if dro_ambig_eps is None:
        dro_ambig_eps = DRO_AMBIG_EPS
    if kappa is None:
        kappa = KAPPA
    if pnorm is None:
        pnorm = PNORM
    if rho_dro is None:
        rho_dro = RHO_DRO
    if tau is None:
        tau = TAU
    if G is None:
        G = G_CONST

    feature_dim = int(client_norm["x0"].shape[1])

    param = {
        "epsilon": float(dro_ambig_eps),
        "kappa": float(kappa),
        "pnorm": pnorm,
        "rho": float(rho_dro),
        "tau": float(tau),
    }

    model = MultiFederatedDRSVM(
        param,
        num_clients=int(G),
        num_classes=int(num_classes),
        feature_dim=feature_dim,
    )
    model.train(client_norm, num_rounds=int(num_rounds))
    return model


In [26]:
def run_f1_benchmark_table(
    datasets,
    seeds,
    eps_dp,
    delta_dp,
    num_rounds,
    include_dp_erm=True,
    include_np_dro=True,
    verbose=True,

    dro_ambig_eps=None,
    kappa=None,
    pnorm=None,
    rho_erm=None,
    rho_dro=None,
    cw=None,
    L_lip=None,
    lambda_reg=None,

    # new DP-DRO args
    C_pen=None,
    gamma_pen=None,
    tau_init=None,
    clambda=None,
    S_lam=None,
    V_lam=None,

    G=None,
):
    dro_ambig_eps = DRO_AMBIG_EPS if dro_ambig_eps is None else dro_ambig_eps
    kappa         = KAPPA         if kappa is None else kappa
    pnorm         = PNORM         if pnorm is None else pnorm
    rho_erm       = RHO_ERM       if rho_erm is None else rho_erm
    rho_dro       = RHO_DRO       if rho_dro is None else rho_dro
    cw            = CW            if cw is None else cw
    L_lip         = L_LIP         if L_lip is None else L_lip
    lambda_reg    = LAMBDA_REG    if lambda_reg is None else lambda_reg

    C_pen         = C_PEN         if C_pen is None else C_pen
    gamma_pen     = GAMMA_PEN     if gamma_pen is None else gamma_pen
    tau_init      = TAU_INIT      if tau_init is None else tau_init
    clambda       = C_LAMBDA      if clambda is None else clambda
    S_lam         = S_LAM         if S_lam is None else S_lam
    V_lam         = V_LAM         if V_lam is None else V_lam

    G             = G_CONST       if G is None else G

    model_list = [("DP DRO SVM", "dp_dro_lin")]
    if include_dp_erm:
        model_list.append(("DP ERM SVM", "dp_erm"))
    if include_np_dro:
        model_list.append(("Non-Private DRO SVM", "np_dro"))

    out = {mn: {} for (mn, _) in model_list}

    for ds in datasets:
        ds_name = ds["name"]
        K = int(ds["num_classes"])

        if verbose:
            print(f"\nDataset {ds_name} num_classes {K}")

        for (model_name, model_key) in model_list:
            scores = []

            for seed in seeds:
                client_norm, test_norm = _split_cache.get(
                    dataset_name=ds_name,
                    seed=seed,
                    split_fn=ds["get_split"],
                )

                if model_key == "dp_dro_lin":
                    model = train_model_dp_dro_linearized(
                        client_norm=client_norm,
                        num_classes=K,
                        seed=seed,
                        eps_dp=eps_dp,
                        delta_dp=delta_dp,
                        num_rounds=num_rounds,
                        dro_ambig_eps=dro_ambig_eps,
                        kappa=kappa,
                        pnorm=pnorm,
                        rho_dro=rho_dro,
                        cw=cw,
                        L_lip=L_lip,
                        C_pen=C_pen,
                        gamma_pen=gamma_pen,
                        tau_init=tau_init,
                        clambda=clambda,
                        S_lam=S_lam,
                        V_lam=V_lam,
                        G=G,
                    )

                elif model_key == "dp_erm":
                    model = train_model_dp_erm(
                        client_norm=client_norm,
                        num_classes=K,
                        seed=seed,
                        eps_dp=eps_dp,
                        delta_dp=delta_dp,
                        num_rounds=num_rounds,
                        rho_erm=rho_erm,
                        lambda_reg=lambda_reg,
                        cw=cw,
                        L_lip=L_lip,
                        G=G,
                    )

                elif model_key == "np_dro":
                    model = train_model_np_dro(
                        client_norm=client_norm,
                        num_classes=K,
                        seed=seed,
                        num_rounds=num_rounds,
                        dro_ambig_eps=dro_ambig_eps,
                        kappa=kappa,
                        pnorm=pnorm,
                        rho_dro=rho_dro,
                        G=G,
                    )
                else:
                    raise ValueError(f"Unknown model_key: {model_key}")

                y_true = np.asarray(test_norm["y"]).astype(int).reshape(-1)

                y_pred = predict_labels(model, test_norm, num_classes=K)
                scores.append(macro_f1_score(y_true, y_pred, num_classes=K))

            mean = float(np.mean(scores))
            std  = float(np.std(scores, ddof=1)) if len(scores) > 1 else 0.0
            out[model_name][ds_name] = (mean, std)

            if verbose:
                print(f"{model_name}: macro F1 {mean:.3f} ± {std:.3f}")

    df = pd.DataFrame(index=[mn for (mn, _) in model_list],
                      columns=[ds["name"] for ds in datasets])
    for mn in df.index:
        for ds_name in df.columns:
            mean, std = out[mn][ds_name]
            df.loc[mn, ds_name] = f"{mean:.3f} ± {std:.3f}"

    return df, out

In [27]:
seeds = [0]

perc_train = 0.8
perc_test  = 0.2
perc_wrong_y = 0.15
perc_class_per_client = 0

G_CONST = 4
client_sample_props = [0.25, 0.25, 0.25, 0.25]

num_rounds = 1

EPS_DP   = 0.5
DELTA_DP = 1e-6

DRO_AMBIG_EPS = 7e-4
KAPPA = 0.5
PNORM = float('Inf')

RHO_ERM = 0.5
RHO_DRO = 0.5

CW = 2.0
L_LIP = 1.0
LAMBDA_REG = 1e-4

# Bertsekas penalty parameters
GAMMA_PEN = 0.1
TAU_INIT  = 0.0

# lambda-block stepsize constants Assuming upperbound on the norm of M is 2
C_LAMBDA = 2 
S_LAM   = KAPPA 
V_LAM   = 1     
C_PEN = 0.4


UCI_ID_1 = 33  # Dermatology
UCI_ID_2 = 53  # Iris
NUM_CLASSES_1 = 6
NUM_CLASSES_2 = 3
UCI_ID_3 = 109 # Wine
NUM_CLASSES_3 = 3

clip_L = 1.0
feature_scaling = "minmax"

CSV_PATH_SIM = r"C:\Users\aphg2\OneDrive\Documents\MATLAB\Examples\R2025b\predmaint\MultiClassFaultDetectionUsingSimDataExample\ML_dataset_features_20260125_121332.csv"
NUM_CLASSES_CSV = 4

CSV_PATH_PENGUINS = r"C:\Users\aphg2\OneDrive\Documents\DRO\penguins.csv"
NUM_CLASSES_PENGUINS = 3

CSV_PATH_SEEDS = r"C:\Users\aphg2\OneDrive\Documents\DRO\seeds_dataset.csv"
NUM_CLASSES_SEEDS = 3

datasets_all = [
    make_dataset_config(
        name=f"UCI {UCI_ID_1}",
        split_fn_seeded=make_get_split_real(
            uci_id=UCI_ID_1,
            num_classes=NUM_CLASSES_1,
            perc_train=perc_train,
            perc_test=perc_test,
            perc_wrong_y=perc_wrong_y,
            perc_class_per_client=perc_class_per_client,
            client_sample_props=client_sample_props,
            G=G_CONST
        ),
        num_classes=NUM_CLASSES_1,
    ),
    make_dataset_config(
        name=f"UCI {UCI_ID_2}",
        split_fn_seeded=make_get_split_real(
            uci_id=UCI_ID_2,
            num_classes=NUM_CLASSES_2,
            perc_train=perc_train,
            perc_test=perc_test,
            perc_wrong_y=perc_wrong_y,
            perc_class_per_client=perc_class_per_client,
            client_sample_props=client_sample_props,
            G=G_CONST
        ),
        num_classes=NUM_CLASSES_2,
    ),
    make_dataset_config(
        name=f"UCI {UCI_ID_3}",
        split_fn_seeded=make_get_split_real(
            uci_id=UCI_ID_3,
            num_classes=NUM_CLASSES_3,
            perc_train=perc_train,
            perc_test=perc_test,
            perc_wrong_y=perc_wrong_y,
            perc_class_per_client=perc_class_per_client,
            client_sample_props=client_sample_props,
            G=G_CONST
        ),
        num_classes=NUM_CLASSES_3,
    ),
    make_dataset_config(
        name="Palmer Penguins",
        split_fn_seeded=make_get_split_csv(
            csv_path=CSV_PATH_PENGUINS,
            num_classes=NUM_CLASSES_PENGUINS,
            perc_train=perc_train,
            perc_test=perc_test,
            perc_wrong_y=perc_wrong_y,
            perc_class_per_client=perc_class_per_client,
            client_sample_props=client_sample_props,
            G=G_CONST,
            label_col="species",
            feature_scaling=feature_scaling,
            feature_cols=None,
            drop_cols=None,
        ),
        num_classes=NUM_CLASSES_PENGUINS,
    ),
    make_dataset_config(
        name="Seeds",
        split_fn_seeded=make_get_split_csv(
            csv_path=CSV_PATH_SEEDS,
            num_classes=NUM_CLASSES_SEEDS,
            perc_train=perc_train,
            perc_test=perc_test,
            perc_wrong_y=perc_wrong_y,
            perc_class_per_client=perc_class_per_client,
            client_sample_props=client_sample_props,
            G=G_CONST,
            label_col="class",
            feature_scaling=feature_scaling,
            feature_cols=None,
            drop_cols=None,
        ),
        num_classes=NUM_CLASSES_SEEDS,
    ),
    make_dataset_config(
        name="Sim CSV",
        split_fn_seeded=make_get_split_csv(
            csv_path=CSV_PATH_SIM,
            num_classes=NUM_CLASSES_CSV,
            perc_train=perc_train,
            perc_test=perc_test,
            perc_wrong_y=perc_wrong_y,
            perc_class_per_client=perc_class_per_client,
            client_sample_props=client_sample_props,
            G=G_CONST,
            label_builder=build_fault_labels_numeric_from_flags,
            feature_scaling=feature_scaling,
            feature_cols=None,
            drop_cols=[
                "LeakFault", "BlockingFault", "BearingFault",
                "LeakFlag", "BlockingFlag", "BearingFlag", "CombinedFlag"
            ],
        ),
        num_classes=NUM_CLASSES_CSV,
    ),
]

datasets_run = datasets_all[:-1]

df_f1, raw = run_f1_benchmark_table(
    datasets=datasets_run,
    seeds=seeds,
    eps_dp=EPS_DP,
    delta_dp=DELTA_DP,
    num_rounds=num_rounds,
    include_dp_erm=True,
    include_np_dro=True,
    verbose=True,
    dro_ambig_eps=DRO_AMBIG_EPS,
    kappa=KAPPA,
    pnorm=PNORM,
    rho_erm=RHO_ERM,
    rho_dro=RHO_DRO,
    cw=CW,
    L_lip=L_LIP,
    lambda_reg=LAMBDA_REG,

    # NEW for updated DP-DRO
    C_pen=C_PEN,
    gamma_pen=GAMMA_PEN,
    tau_init=TAU_INIT,
    clambda=C_LAMBDA,
    S_lam=S_LAM,
    V_lam=V_LAM,

    G=G_CONST,
)

display(df_f1)



Dataset UCI 33 num_classes 6
[UCI 33 raw] Dropping 8 / 366 rows with NaN/Inf
Class distribution BEFORE split: {0: 111, 1: 60, 2: 71, 3: 48, 4: 48, 5: 20}
 n_train: {0: 89, 1: 48, 2: 57, 3: 39, 4: 39, 5: 16}, n_test: {0: 22, 1: 12, 2: 14, 3: 9, 4: 9, 5: 4}

Class 0 - Test Samples:
Original Index: 3 | Original Label: 0 | x_test[0]: [ 2.  3.  2.  0.  1.  0.  0.  0.  0.  1.  0.  0.  0.  0.  0.  0.  2.  0.
  2.  3.  3.  2.  0.  2.  0.  2.  0.  0.  0.  0.  0.  2.  0. 43.] | y_test[0]: 0.0
Original Index: 5 | Original Label: 0 | x_test[1]: [ 3.  1.  2.  1.  0.  0.  0.  0.  2.  3.  0.  0.  0.  0.  0.  0.  2.  0.
  2.  3.  2.  2.  0.  3.  0.  2.  0.  0.  0.  0.  0.  2.  0. 17.] | y_test[1]: 0.0
Original Index: 7 | Original Label: 0 | x_test[2]: [ 2.  1.  1.  0.  0.  0.  0.  0.  1.  1.  1.  0.  0.  1.  0.  0.  2.  2.
  2.  2.  2.  2.  0.  1.  0.  1.  0.  0.  0.  0.  0.  1.  0. 40.] | y_test[2]: 0.0
Original Index: 10 | Original Label: 0 | x_test[3]: [ 3.  3.  2.  2.  1.  0.  0.  0.  0.  1.  0. 

Original Index: 101 | Original Label: 5 | x_rem[272]: [ 2.  2.  1.  1.  0.  0.  2.  0.  2.  0.  1.  0.  0.  0.  0.  1.  1.  1.
  1.  0.  0.  0.  0.  0.  0.  0.  0.  2.  0.  1.  2.  2.  0. 10.] | y_rem[272]: 5.0
Original Index: 120 | Original Label: 5 | x_rem[273]: [1. 1. 2. 0. 0. 0. 3. 0. 3. 0. 1. 0. 0. 0. 0. 2. 1. 1. 1. 1. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 2. 2. 2. 0. 7.] | y_rem[273]: 5.0
Original Index: 125 | Original Label: 5 | x_rem[274]: [2. 2. 1. 0. 0. 0. 2. 0. 2. 0. 0. 0. 0. 0. 0. 3. 2. 0. 1. 0. 0. 0. 0. 0.
 0. 0. 0. 3. 0. 2. 2. 2. 0. 7.] | y_rem[274]: 5.0
Original Index: 133 | Original Label: 5 | x_rem[275]: [ 2.  2.  0.  1.  0.  0.  2.  0.  2.  0.  0.  0.  0.  0.  0.  2.  2.  0.
  0.  0.  0.  0.  0.  0.  0.  0.  0.  1.  0.  2.  2.  2.  0. 22.] | y_rem[275]: 5.0
Original Index: 157 | Original Label: 5 | x_rem[276]: [2. 2. 2. 0. 0. 0. 2. 0. 2. 0. 0. 0. 0. 0. 0. 2. 2. 2. 2. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 2. 2. 2. 0. 8.] | y_rem[276]: 5.0
Original Index: 158 | Original Label: 5 | x_rem

Set parameter LicenseID to value 2767255
Academic license - for non-commercial use only - expires 2027-01-18
[client 0, round 1] status = 2
[client 1, round 1] status = 2
[client 2, round 1] status = 2
[client 3, round 1] status = 2
[LIN-DRO] round 1: ||w^k - w^0||_F = 6.3786e-01
Pred counts: [ 0  0 66  4  0  0]
DP DRO SVM: macro F1 0.058 ± 0.000
DP ERM SVM: macro F1 0.084 ± 0.000
Weight change norm (round 0): 0.31515439107471843
Non-Private DRO SVM: macro F1 0.882 ± 0.000

Dataset UCI 53 num_classes 3
Class distribution BEFORE split: {0: 50, 1: 50, 2: 50}
 n_train: {0: 40, 1: 40, 2: 40}, n_test: {0: 10, 1: 10, 2: 10}

Class 0 - Test Samples:
Original Index: 1 | Original Label: 0 | x_test[0]: [5.7 3.8 1.7 0.3] | y_test[0]: 0.0
Original Index: 5 | Original Label: 0 | x_test[1]: [5.4 3.4 1.5 0.4] | y_test[1]: 0.0
Original Index: 11 | Original Label: 0 | x_test[2]: [4.8 3.  1.4 0.1] | y_test[2]: 0.0
Original Index: 12 | Original Label: 0 | x_test[3]: [5.5 3.5 1.3 0.2] | y_test[3]: 0.0
Ori

Non-Private DRO SVM: macro F1 0.509 ± 0.000

Dataset UCI 109 num_classes 3
Class distribution BEFORE split: {0: 59, 1: 71, 2: 48}
 n_train: {0: 48, 1: 57, 2: 39}, n_test: {0: 11, 1: 14, 2: 9}

Class 0 - Test Samples:
Original Index: 0 | Original Label: 0 | x_test[0]: [1.364e+01 3.100e+00 2.560e+00 1.520e+01 1.160e+02 2.700e+00 3.030e+00
 1.700e-01 1.660e+00 5.100e+00 9.600e-01 3.360e+00 8.450e+02] | y_test[0]: 0.0
Original Index: 1 | Original Label: 0 | x_test[1]: [1.421e+01 4.040e+00 2.440e+00 1.890e+01 1.110e+02 2.850e+00 2.650e+00
 3.000e-01 1.250e+00 5.240e+00 8.700e-01 3.330e+00 1.080e+03] | y_test[1]: 0.0
Original Index: 3 | Original Label: 0 | x_test[2]: [1.373e+01 1.500e+00 2.700e+00 2.250e+01 1.010e+02 3.000e+00 3.250e+00
 2.900e-01 2.380e+00 5.700e+00 1.190e+00 2.710e+00 1.285e+03] | y_test[2]: 0.0
Original Index: 5 | Original Label: 0 | x_test[3]: [1.43e+01 1.92e+00 2.72e+00 2.00e+01 1.20e+02 2.80e+00 3.14e+00 3.30e-01
 1.97e+00 6.20e+00 1.07e+00 2.65e+00 1.28e+03] | y_test[

Weight change norm (round 0): 0.284905100608603
Non-Private DRO SVM: macro F1 0.910 ± 0.000

Dataset Palmer Penguins num_classes 3
Class distribution BEFORE split: {0: 146, 1: 68, 2: 119}
 n_train: {0: 117, 1: 55, 2: 96}, n_test: {0: 29, 1: 13, 2: 23}
Train class counts used for splitting: {0: 117, 1: 55, 2: 96}
Per class per client:{0: {0: 0.25, 1: 0.25, 2: 0.25}, 1: {0: 0.25, 1: 0.25, 2: 0.25}, 2: {0: 0.25, 1: 0.25, 2: 0.25}, 3: {0: 0.25, 1: 0.25, 2: 0.25}}
Class 0: Total available samples: 117
Class 0: Preliminary allocations: [29, 29, 29, 29]
Class 0: Total allocated after floor: 116
Class 0: Discrepancy: 1
Class 0: Final allocations per client: [30, 29, 29, 29]
Class 1: Total available samples: 55
Class 1: Preliminary allocations: [13, 13, 13, 13]
Class 1: Total allocated after floor: 52
Class 1: Discrepancy: 3
Class 1: Final allocations per client: [14, 14, 14, 13]
Class 2: Total available samples: 96
Class 2: Preliminary allocations: [24, 24, 24, 24]
Class 2: Total allocated aft

,UCI 33,UCI 53,UCI 109,Palmer Penguins,Seeds
DP DRO SVM,0.058 ± 0.000,0.167 ± 0.000,0.140 ± 0.000,0.198 ± 0.000,0.268 ± 0.000
DP ERM SVM,0.084 ± 0.000,0.167 ± 0.000,0.340 ± 0.000,0.421 ± 0.000,0.093 ± 0.000
Non-Private DRO SVM,0.882 ± 0.000,0.509 ± 0.000,0.910 ± 0.000,0.430 ± 0.000,0.755 ± 0.000
